In [11]:
import glob
import os
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split



In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
base_path = "/content/drive/MyDrive/dev_phase/subtask1/train/"
files = glob.glob(os.path.join(base_path, "*.csv"))

data = {}

for file in files:
    lang = os.path.splitext(os.path.basename(file))[0]  # amh, arb, eng

    df = pd.read_csv(file)

    data[lang] = {
        "X": df["text"].tolist(),
        "y": df["polarization"].tolist(),
        "df": df
    }

print("Loaded languages:", sorted(data.keys()))


Loaded languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'ita', 'khm', 'mya', 'nep', 'ori', 'pan', 'pol', 'rus', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']


In [43]:
model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
embedding_model = AutoModel.from_pretrained(model_name).to(device)
embedding_model.eval()

for p in embedding_model.parameters():
    p.requires_grad = False

def mean_pooling(model_output, attention_mask):
    token_embeds = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    sum_embeddings = torch.sum(token_embeds * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [44]:
def get_all_embeddings(texts, model, tokenizer, device, batch_size=32):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**enc)
            embeddings = mean_pooling(outputs, enc['attention_mask'])
        all_embs.append(embeddings.cpu())
    return torch.cat(all_embs, dim=0)


embeddings_by_lang = {}

for lang, content in data.items():   # data[lang] has "X" and "y"
    print(f"Embedding language: {lang}")

    X_text = content["X"]
    y_labels = content["y"]

    X_emb = get_all_embeddings(X_text, embedding_model, tokenizer, device)
    y_tensor = torch.tensor(y_labels, dtype=torch.long)

    embeddings_by_lang[lang] = {
        "X": X_emb,
        "y": y_tensor
    }


Embedding language: urd


Embedding: 100%|██████████| 112/112 [00:33<00:00,  3.35it/s]


Embedding language: fas


Embedding: 100%|██████████| 103/103 [00:34<00:00,  2.99it/s]


Embedding language: ori


...

Embedding: 100%|██████████| 105/105 [00:52<00:00,  2.01it/s]


Embedding language: eng


Embedding: 100%|██████████| 101/101 [00:29<00:00,  3.48it/s]


Embedding language: mya


Embedding: 100%|██████████| 91/91 [01:04<00:00,  1.40it/s]


In [45]:
class CNNClassifier(nn.Module):
    def __init__(self, embed_dim, num_classes):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear((embed_dim // 4) * 256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: [B, embed_dim]
        x = x.unsqueeze(1)        # [B, 1, embed_dim]
        x = self.conv(x)          # [B, 256, embed_dim/4]
        x = x.flatten(1)          # [B, *]
        return self.fc(x)


In [46]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


In [47]:
def evaluate_language_cv(X_embeddings, y_tensor, device, num_classes, k=5, epochs=5, batch_size=32):
    """
    Evaluate one language using Stratified K-Fold CV.
    Returns mean and std for Accuracy and Macro F1.
    """
    X = X_embeddings.numpy()
    y = y_tensor.numpy()

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    all_acc = []
    all_macro_f1 = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        # Split
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Tensors
        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.long)
        X_test_t = torch.tensor(X_test, dtype=torch.float32)
        y_test_t = torch.tensor(y_test, dtype=torch.long)

        # DataLoader
        train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)

        # Initialize classifier
        classifier = CNNClassifier(X_train_t.shape[1], num_classes).to(device)
        optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        total, trainable = count_params(classifier)
        print(f"[Fold {fold+1}] Total params: {total:,} | Trainable: {trainable:,}")

        # Train
        classifier.train()
        for epoch in range(epochs):
            for bx, by in train_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                logits = classifier(bx)
                loss = criterion(logits, by)
                loss.backward()
                optimizer.step()

        # Evaluate
        classifier.eval()
        with torch.no_grad():
            logits = classifier(X_test_t.to(device))
            y_pred = torch.argmax(logits, dim=1).cpu().numpy()

        acc = np.mean(y_pred == y_test)
        macro_f1 = f1_score(y_test, y_pred, average='macro')

        all_acc.append(acc)
        all_macro_f1.append(macro_f1)

    return np.mean(all_acc), np.std(all_acc), np.mean(all_macro_f1), np.std(all_macro_f1)


In [48]:
results = []

# Assuming you have embeddings already: embeddings_by_lang
# embeddings_by_lang[lang] = {"X": X_embeddings, "y": y_tensor}
num_classes = len(np.unique(y_tensor))  # assuming same number of classes for all languages

for lang, content in embeddings_by_lang.items():
    print(f"\nEvaluating language: {lang}")

    X_emb = content["X"]
    y_tensor = content["y"]

    mean_acc, std_acc, mean_f1, std_f1 = evaluate_language_cv(
        X_emb, y_tensor, device, num_classes, k=5, epochs=5, batch_size=32
    )

    results.append({
        "language": lang,
        "mean_acc": mean_acc,
        "std_acc": std_acc,
        "mean_macro_f1": mean_f1,
        "std_macro_f1": std_f1
    })


results_df = pd.DataFrame(results)
results_df = results_df.sort_values("mean_macro_f1", ascending=False)
results_df.to_csv("e5_cnn.csv", index=False)
print(results_df)



Evaluating language: urd
[Fold 1] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 2] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 3] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 4] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 5] Total params: 16,877,058 | Trainable: 16,877,058

Evaluating language: fas
[Fold 1] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 2] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 3] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 4] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 5] Total params: 16,877,058 | Trainable: 16,877,058

Evaluating language: ori
[Fold 1] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 2] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 3] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 4] Total params: 16,877,058 | Trainable: 16,877,058
[Fold 5] Total params: 16,877,058 | Trainable: 16,877,058

Evaluating language: arb
[Fold 1] Total params: 16,